# Source Exploration and Data Contracts

Run from the repository root:

`jupyter execute scratch/explore_source_data.ipynb`

The notebook uses only `pandas`, `numpy`, `json`, `csv`, `re`, `shapely`, and `nbformat`/Jupyter runtime support.

In [ ]:
import csv
import json
import math
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from shapely.geometry import Point, shape

base = Path.cwd().resolve()
if base.name == 'scratch':
    base = base.parent
kamernet_path = base / 'data/rentals.json'
airbnb_path = base / 'data/airbnb.csv'
postcode_path = base / 'data/geo/post_codes.geojson'
area_path = base / 'data/geo/amsterdam_areas.geojson'

In [ ]:
with kamernet_path.open(encoding='utf-8') as f:
    kamernet = json.load(f)
airbnb = pd.read_csv(airbnb_path)
print({'kamernet_rows': len(kamernet), 'airbnb_rows': len(airbnb)})
print('Kamernet keys:', sorted(set().union(*(row.keys() for row in kamernet))))
print('Airbnb dtypes:\n', airbnb.dtypes)
print('Airbnb nulls:\n', airbnb.isna().sum())

In [ ]:
def kam_id(row):
    value = row.get('_id')
    return str(value[0]) if isinstance(value, list) and value else str(value)

def normalize_postcode(value):
    if value is None:
        return None
    normalized = re.sub(r'[^A-Za-z0-9]', '', str(value)).upper()
    return normalized if re.fullmatch(r'\d{4}[A-Z]{2}', normalized) else None

amsterdam = [row for row in kamernet if str(row.get('city', '')).strip().lower() == 'amsterdam']
active = [row for row in amsterdam if row.get('isRoomActive') == 'true']
inactive = [row for row in amsterdam if row.get('isRoomActive') == 'false']
kamernet_duplicate_ids = len(kamernet) - len({kam_id(row) for row in kamernet})
airbnb_duplicate_rows = len(airbnb) - len(airbnb.drop_duplicates())

print({'amsterdam': len(amsterdam), 'active': len(active), 'inactive': len(inactive),
       'unknown_active': len(amsterdam) - len(active) - len(inactive),
       'kamernet_duplicate_ids': kamernet_duplicate_ids,
       'airbnb_duplicate_exact_rows': airbnb_duplicate_rows})

In [ ]:
postcode_normalized = [normalize_postcode(row.get('postalCode')) for row in amsterdam]
postcode_normalized = [value for value in postcode_normalized if value]
print({'Amsterdam valid Kamernet postcode rows': len(postcode_normalized),
       'Amsterdam distinct PC4': len({value[:4] for value in postcode_normalized})})

raw_zips = airbnb['zipcode'].fillna('').astype(str).str.strip()
zip_formats = raw_zips.map(lambda value: (
    'missing' if not value else
    'pc4' if re.fullmatch(r'\d{4}', value) else
    'full6' if re.fullmatch(r'\d{4}\s*[A-Z]{2}', value.upper()) else
    'invalid'))
print('Airbnb zipcode formats:\n', zip_formats.value_counts().to_string())
print('Invalid zipcode examples:', raw_zips[zip_formats == 'invalid'].head(20).tolist())
print('Airbnb coordinate bounds:', {
    'latitude': [airbnb['latitude'].min(), airbnb['latitude'].max()],
    'longitude': [airbnb['longitude'].min(), airbnb['longitude'].max()]})

In [ ]:
rent_values = []
invalid_rents = []
for row in amsterdam:
    match = re.search(r'(\d+(?:[.,]\d+)?)', str(row.get('rent', '')))
    if match:
        rent_values.append(float(match.group(1).replace(',', '.')))
    else:
        invalid_rents.append(row)

prices = pd.to_numeric(airbnb['price'], errors='coerce')
print({'Amsterdam rent rows parseable': len(rent_values),
       'Amsterdam rent invalid': len(invalid_rents),
       'Kamernet rent range': [min(rent_values), max(rent_values)],
       'Airbnb price invalid': int(prices.isna().sum()),
       'Airbnb price range': [float(prices.min()), float(prices.max())]})
print('Optional Airbnb gaps:', {
    'bedrooms': int(airbnb['bedrooms'].isna().sum()),
    'review_scores_value': int(airbnb['review_scores_value'].isna().sum())})

In [ ]:
with postcode_path.open(encoding='utf-8') as f:
    postcode_geo = json.load(f)
with area_path.open(encoding='utf-8') as f:
    area_geo = json.load(f)

polygons = [(str(feature['properties']['pc4_code']).strip(), shape(feature['geometry']))
            for feature in postcode_geo['features']]
recovery = Counter()
for row in airbnb.itertuples(index=False):
    raw = '' if pd.isna(row.zipcode) else str(row.zipcode).strip()
    normalized = re.sub(r'[^A-Za-z0-9]', '', raw).upper()
    if re.fullmatch(r'\d{4}(?:[A-Z]{2})?', normalized):
        continue
    point = Point(float(row.longitude), float(row.latitude))
    hits = sorted({code for code, polygon in polygons if polygon.contains(point)})
    if len(hits) == 1:
        recovery['recovered_unique'] += 1
    elif not hits:
        recovery['unknown_outside_polygons'] += 1
    else:
        recovery['ambiguous_multiple'] += 1

print('Postcode recovery:', dict(recovery))
print({'postcode_features': len(postcode_geo['features']),
       'area_features': len(area_geo['features']),
       'postcode_property_keys': sorted(set().union(*(feature['properties'].keys() for feature in postcode_geo['features']))),
       'area_property_keys': sorted(set().union(*(feature['properties'].keys() for feature in area_geo['features'])))})